In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import os

# ==========================================
# 1. KONFIGURASI PATH (Sesuai Dataset Kamu)
# ==========================================
# Path root yang berisi folder TB dan NTB
DATA_PATH = '/kaggle/input/zip-spect/kaggle/working/spectrograms'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Cek apakah folder terbaca
if os.path.exists(DATA_PATH):
    print(f"✅ Folder ditemukan! Isi: {os.listdir(DATA_PATH)}")
else:
    print("❌ Path salah! Cek kembali lokasi dataset di sidebar kiri.")

# ==========================================
# 2. DATA LOADER (Auto-Split)
# ==========================================
print("\nMemuat dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# ==========================================
# 3. ARSITEKTUR ULTIMATE (MobileNetV2 Fine-Tuned)
# ==========================================
print("\nMenyusun Arsitektur Model...")

# Augmentasi untuk variasi data agar AUC tinggi
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.02),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), 
    include_top=False, 
    weights='imagenet'
)

# Fine-tuning: Buka layer setelah index 100
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# ==========================================
# 4. CALLBACKS & TRAINING
# ==========================================
cbs = [
    callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3, mode='max', verbose=1),
    callbacks.EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True),
    callbacks.ModelCheckpoint('Best_TBC_Model_Competition.keras', monitor='val_auc', save_best_only=True, mode='max')
]

print("\n🚀 Memulai Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=cbs
)

# ==========================================
# 5. SAVING FINAL MODEL
# ==========================================
model.save('RespiScan_Final_Model.h5')
print("\n✅ SELESAI! Model terbaik disimpan sebagai 'Best_TBC_Model_Competition.keras'")

2026-01-30 13:32:57.469220: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769779977.670731      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769779977.724723      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769779978.187200      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769779978.187236      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769779978.187239      24 computation_placer.cc:177] computation placer alr

✅ Folder ditemukan! Isi: ['TB', 'NTB']

Memuat dataset...
Found 8823 files belonging to 2 classes.
Using 7059 files for training.


I0000 00:00:1769779997.585582      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 8823 files belonging to 2 classes.
Using 1764 files for validation.

Menyusun Arsitektur Model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

🚀 Memulai Training...
Epoch 1/35


I0000 00:00:1769780017.324477      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


221/221 ━━━━━━━━━━━━━━━━━━━━ 36s 89ms/step - accuracy: 0.6111 - auc: 0.6292 - loss: 1.1786 - val_accuracy: 0.6423 - val_auc: 0.6674 - val_loss: 1.0895 - learning_rate: 1.0000e-04
Epoch 2/35
221/221 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - accuracy: 0.6962 - auc: 0.7588 - loss: 1.0088 - val_accuracy: 0.6729 - val_auc: 0.7176 - val_loss: 1.0655 - learning_rate: 1.0000e-04
Epoch 3/35
221/221 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - accuracy: 0.7569 - auc: 0.8235 - loss: 0.9144 - val_accuracy: 0.6939 - val_auc: 0.7573 - val_loss: 1.0448 - learning_rate: 1.0000e-04
Epoch 4/35
221/221 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - accuracy: 0.7869 - auc: 0.8664 - loss: 0.8349 - val_accuracy: 0.6973 - val_auc: 0.7922 - val_loss: 1.0596 - learning_rate: 1.0000e-04
Epoch 5/35
221/221 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - accuracy: 0.8108 - auc: 0.8963 - loss: 0.7710 - val_accuracy: 0.7557 - val_auc: 0.8475 - val_loss: 0.8334 - learning_rate: 1.0000e-04
Epoch 6/35
221/221 ━━━━━━━━━━━━━━━━━━━━ 17s 75ms/step - accur


✅ SELESAI! Model terbaik disimpan sebagai 'Best_TBC_Model_Competition.keras'
